# Cell 0 — Mount + Imports


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, json, glob
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb

Mounted at /content/drive


# Cell 1 — Paths

In [ ]:
EXPORT_ROOT = "/content/drive/MyDrive/Dataset/Model1_Results/step10_exports"
DATA_PATH   = "/content/drive/MyDrive/Dataset/Dataset_plus5_2years_45stations_REALISTIC_SA_PRODUCTION_FINAL_B.csv"  # مثال

SPLIT_TAG = "station_holdout"  # أو: year_holdout / station_time_holdout / city_holdout

print("EXPORT_ROOT:", EXPORT_ROOT)
print("DATA_PATH:", DATA_PATH)
print("SPLIT_TAG:", SPLIT_TAG)

EXPORT_ROOT: /content/drive/MyDrive/Dataset/Model1_Results/step10_exports
DATA_PATH: /content/drive/MyDrive/Dataset/Dataset_plus5_2years_45stations_REALISTIC_SA_PRODUCTION_FINAL_B.csv
SPLIT_TAG: station_holdout


# Cell 2 — File Finder

In [ ]:
def find_artifact(export_root, split_tag, filename_pattern):

    candidates = []

    candidates += glob.glob(os.path.join(export_root, split_tag, filename_pattern))
    candidates += glob.glob(os.path.join(export_root, filename_pattern))
    candidates += glob.glob(os.path.join(export_root, "**", filename_pattern), recursive=True)

    prefer = [c for c in candidates if split_tag in os.path.basename(c)]
    if prefer:
        return sorted(prefer)[-1]
    if candidates:
        return sorted(candidates)[-1]
    return None


MODEL_PATH   = find_artifact(EXPORT_ROOT, SPLIT_TAG, f"xgb_{SPLIT_TAG}*.json")
FEATS_PATH   = find_artifact(EXPORT_ROOT, SPLIT_TAG, f"feature_cols_{SPLIT_TAG}*.pkl")
DRIVERS_PATH = find_artifact(EXPORT_ROOT, SPLIT_TAG, f"global_drivers_top10_{SPLIT_TAG}*.json")

print("MODEL_PATH:", MODEL_PATH)
print("FEATS_PATH:", FEATS_PATH)
print("DRIVERS_PATH:", DRIVERS_PATH)

assert MODEL_PATH is not None, "❌ xgb_*.json"
assert FEATS_PATH is not None, "❌ feature_cols_*.pkl"


MODEL_PATH: /content/drive/MyDrive/Dataset/Model1_Results/step10_exports/xgb_station_holdout.json
FEATS_PATH: /content/drive/MyDrive/Dataset/Model1_Results/step10_exports/station_holdout/feature_cols_station_holdout.pkl
DRIVERS_PATH: /content/drive/MyDrive/Dataset/Model1_Results/step10_exports/station_holdout/global_drivers_top10_station_holdout.json


# Cell 3 — Load the main production model

In [ ]:
# Load xgb model
model = xgb.Booster()
model.load_model(MODEL_PATH)

# Load feature columns used in training (order matters)
feature_cols = joblib.load(FEATS_PATH)
feature_cols = list(feature_cols)

# Load global drivers (optional)
TOP_DRIVERS = None
if DRIVERS_PATH is not None and os.path.exists(DRIVERS_PATH):
    with open(DRIVERS_PATH, "r") as f:
        drivers = json.load(f)
    # drivers: list of dicts {feature, gain}
    TOP_DRIVERS = [d["feature"] for d in drivers[:10] if "feature" in d]

print("Loaded model + feature_cols:", len(feature_cols))
print("Loaded TOP_DRIVERS:", TOP_DRIVERS[:5] if TOP_DRIVERS else None)

Loaded model + feature_cols: 108
Loaded TOP_DRIVERS: ['total_sales', 'power_status_outage', 'event_type_none', 'pumps_working', 'avg_rating']


# Cell 4 — Load dataset (df) + basic cleaning

In [ ]:
df = pd.read_csv(DATA_PATH)

# Ensure date type if exists
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Ensure station_id string
if "station_id" in df.columns:
    df["station_id"] = df["station_id"].astype(str).str.strip()

print("df shape:", df.shape)
df.head(2)

df shape: (90000, 51)


,station_id,city,station_type,area_m2,competitors_within_3km,nearby_services_density,has_convenience_store,has_cafe,has_atm,has_car_wash,...,queue_time_avg,complaints_count,complaint_type,avg_rating,inventory_stockout,inventory_days_of_cover,actual_inventory_usage,performance_score,day_of_week,customer_flow_index
0,S021,Dammam,Urban,1958,7,3.13,1,1,0,0,...,4.46,0,none,4.25,0,4.7,1516.65,90.632594,0,1.13
1,S060,Buraydah,Urban,1564,7,3.16,1,0,1,1,...,3.21,0,none,4.33,0,3.6,651.74,93.671286,4,0.91


# Cell 5 — Build X for inference

In [ ]:
# columns that shouldn't be used as inputs if present
DROP_COLS_BASE = [
    "performance_score",
    "performance_score_obs",
    "shutdown_reason",
]

# leak-like features
LEAK_LIKE_COLS = [
    "is_shutdown",
    "event_duration_minutes",
    "downtime_minutes",
    "event_severity",
    "complaints_count",
    "queue_time_avg",
    "pos_uptime_pct",
    "network_uptime_pct",
]

def build_X_for_inference(raw_df, feature_cols):
    X = raw_df.copy()
    drop_cols = [c for c in DROP_COLS_BASE + LEAK_LIKE_COLS if c in X.columns]
    X = X.drop(columns=drop_cols, errors="ignore")

    # one-hot
    X = pd.get_dummies(X)

    # align to training features
    X = X.reindex(columns=feature_cols, fill_value=0)

    return X

sample_df = df.sort_values("date").tail(200).copy() if "date" in df.columns else df.tail(200).copy()

X_inf = build_X_for_inference(sample_df, feature_cols)
print("X_inf shape:", X_inf.shape)

X_inf shape: (200, 108)


# Cell 6 — Predict + SHAP-like contributions

In [ ]:
def predict_with_contribs(model, X_aligned):
    dmat = xgb.DMatrix(X_aligned, feature_names=list(X_aligned.columns))
    preds = model.predict(dmat)
    contrib = model.predict(dmat, pred_contribs=True)  # (n, num_features+1)
    shap_vals = contrib[:, :-1]  # drop bias term
    return preds, shap_vals

preds, shap_vals = predict_with_contribs(model, X_inf)

print("preds shape:", preds.shape)
print("shap_vals shape:", shap_vals.shape)

preds shape: (200,)
shap_vals shape: (200, 108)


# Cell 7 — Recommendation mapping

In [ ]:
RECOMMENDATION_MAP = {
    "staff_count": "Optimize staff allocation during busy hours.",
    "transactions": "Improve checkout flow and POS efficiency to handle higher transactions.",
    "queue_length_avg": "Reduce queue times by optimizing staffing and pump allocation.",
    "inventory_stockout": "Reduce stockouts by improving inventory replenishment and safety stock levels.",
    "network_quality_index": "Investigate network infrastructure to reduce disruptions.",
}

def dynamic_fallback_action(feature_name: str) -> str:
    f = feature_name.lower()
    clean_name = feature_name.replace("_", " ")

    if "downtime" in f or "shutdown" in f:
        return f"Reduce downtime related to {clean_name} by preventive maintenance and rapid incident response."
    if "complaint" in f:
        return f"Address customer complaints impacting {clean_name} via service quality checks and feedback loops."
    if "queue" in f or "wait" in f:
        return f"Reduce waiting impact of {clean_name} by optimizing staffing and peak-hour operations."
    if "staff" in f or "employee" in f:
        return f"Optimize workforce planning related to {clean_name}."
    if "inventory" in f or "stock" in f:
        return f"Improve inventory management affecting {clean_name} (reorder points, safety stock)."
    if "traffic" in f or "flow" in f:
        return f"Analyze traffic patterns affecting {clean_name} and adjust staffing/promotions accordingly."
    if "sales" in f or "revenue" in f or "volume" in f:
        return f"Review performance drivers related to {clean_name} and optimize pricing/promotions and availability."
    if "area" in f or "layout" in f:
        return f"Evaluate station layout and efficiency linked to {clean_name}."
    if "pump" in f:
        return f"Check pump availability/maintenance affecting {clean_name} and optimize pump utilization."
    return f"Investigate operational impact of {clean_name} and apply corrective measures."

def generate_recommendations(neg_features):
    recs = []
    for f in neg_features:
        if f in RECOMMENDATION_MAP:
            action = RECOMMENDATION_MAP[f]
        else:
            action = dynamic_fallback_action(f)

        recs.append({"feature": f, "action": action})
    return recs

# Cell 8 — Build final results table (Top 3 negative drivers + actions)

In [ ]:
def top_negative_drivers_for_row(shap_row, feature_names, top_k=3, restrict_to=None):
    """
    shap_row: array (num_features,)
    Negative drivers = most negative SHAP contributions
    restrict_to: list of feature names (optional)
    """
    s = pd.Series(shap_row, index=feature_names)

    if restrict_to is not None:
        restrict_to = [f for f in restrict_to if f in s.index]
        if len(restrict_to) > 0:
            s = s.loc[restrict_to]

    neg = s.sort_values(ascending=True).head(top_k)  # most negative
    return [{"feature": idx, "impact": float(val)} for idx, val in neg.items()]

results = []
for i in range(len(sample_df)):
    negs = top_negative_drivers_for_row(
        shap_vals[i],
        feature_names=X_inf.columns,
        top_k=3,
        restrict_to=TOP_DRIVERS
    )
    recs = generate_recommendations([d["feature"] for d in negs])

    results.append({
        "station_id": sample_df["station_id"].iloc[i] if "station_id" in sample_df.columns else None,
        "date": sample_df["date"].iloc[i] if "date" in sample_df.columns else None,
        "time_slot": sample_df["time_slot"].iloc[i] if "time_slot" in sample_df.columns else None,
        "predicted_performance": float(preds[i]),
        "top_negative_drivers": negs,
        "recommended_actions": recs,
    })

results_df = pd.DataFrame(results)

# ---- Flatten for dashboard columns ----
flat = results_df.copy()

for k in [1, 2, 3]:
    flat[f"neg_feature_{k}"] = flat["top_negative_drivers"].apply(lambda xs: xs[k-1]["feature"] if len(xs) >= k else None)
    flat[f"neg_impact_{k}"]  = flat["top_negative_drivers"].apply(lambda xs: xs[k-1]["impact"]  if len(xs) >= k else None)
    flat[f"action_feature_{k}"] = flat["recommended_actions"].apply(lambda xs: xs[k-1]["feature"] if len(xs) >= k else None)
    flat[f"action_text_{k}"]    = flat["recommended_actions"].apply(lambda xs: xs[k-1]["action"]  if len(xs) >= k else None)

flat.head()

,station_id,date,time_slot,predicted_performance,top_negative_drivers,recommended_actions,neg_feature_1,neg_impact_1,action_feature_1,action_text_1,neg_feature_2,neg_impact_2,action_feature_2,action_text_2,neg_feature_3,neg_impact_3,action_feature_3,action_text_3
0,S071,2025-12-30,morning,85.127739,"[{'feature': 'power_status_outage', 'impact': ...","[{'feature': 'power_status_outage', 'action': ...",power_status_outage,0.015871,power_status_outage,Investigate operational impact of power status...,inventory_stockout,0.088297,inventory_stockout,Reduce stockouts by improving inventory replen...,shutdown_type_partial,0.104433,shutdown_type_partial,Reduce downtime related to shutdown type parti...
1,S091,2025-12-30,night,80.564720,"[{'feature': 'staff_count', 'impact': -2.58443...","[{'feature': 'staff_count', 'action': 'Optimiz...",staff_count,-2.584438,staff_count,Optimize staff allocation during busy hours.,power_status_outage,0.015456,power_status_outage,Investigate operational impact of power status...,inventory_stockout,0.084463,inventory_stockout,Reduce stockouts by improving inventory replen...
2,S037,2025-12-30,noon,85.583832,"[{'feature': 'power_status_outage', 'impact': ...","[{'feature': 'power_status_outage', 'action': ...",power_status_outage,0.015768,power_status_outage,Investigate operational impact of power status...,inventory_stockout,0.083799,inventory_stockout,Reduce stockouts by improving inventory replen...,shutdown_type_partial,0.105490,shutdown_type_partial,Reduce downtime related to shutdown type parti...
3,S074,2025-12-30,morning,78.957260,"[{'feature': 'staff_count', 'impact': -3.62435...","[{'feature': 'staff_count', 'action': 'Optimiz...",staff_count,-3.624353,staff_count,Optimize staff allocation during busy hours.,power_status_outage,0.015295,power_status_outage,Investigate operational impact of power status...,actual_inventory_usage,0.021909,actual_inventory_usage,Improve inventory management affecting actual ...
4,S021,2025-12-30,noon,69.930099,"[{'feature': 'event_type_none', 'impact': -9.1...","[{'feature': 'event_type_none', 'action': 'Inv...",event_type_none,-9.186746,event_type_none,Investigate operational impact of event type n...,power_status_outage,0.014536,power_status_outage,Investigate operational impact of power status...,staff_count,0.070201,staff_count,Optimize staff allocation during busy hours.


# Cell 9 - Save outputs

In [ ]:
OUT_DIR = os.path.join(EXPORT_ROOT, "model2_recommendations")
os.makedirs(OUT_DIR, exist_ok=True)

csv_path  = os.path.join(OUT_DIR, f"recommendations_{SPLIT_TAG}.csv")
json_path = os.path.join(OUT_DIR, f"recommendations_{SPLIT_TAG}.json")

flat.to_csv(csv_path, index=False)
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2, default=str)

print("Saved CSV :", csv_path)
print("Saved JSON:", json_path)

Saved CSV : /content/drive/MyDrive/Dataset/Model1_Results/step10_exports/model2_recommendations/recommendations_station_holdout.csv
Saved JSON: /content/drive/MyDrive/Dataset/Model1_Results/step10_exports/model2_recommendations/recommendations_station_holdout.json
